In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

print("✓ imports ready")


✓ imports ready


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

base_url = "https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{}.parquet"

seasons = [2022, 2023, 2024, 2025]
dfs = []

for season in seasons:
    print(f"Loading {season}...")
    url = base_url.format(season)
    df = pd.read_parquet(url)
    dfs.append(df)

pbp = pd.concat(dfs, ignore_index=True)
print(f"✓ Loaded {len(pbp):,} plays across 4 seasons")

Loading 2022...
Loading 2023...
Loading 2024...
Loading 2025...
✓ Loaded 197,362 plays across 4 seasons


In [3]:
# Keep only the columns we need
cols = ['game_id', 'posteam', 'defteam', 'game_seconds_remaining',
        'score_differential', 'down', 'ydstogo', 'yardline_100',
        'posteam_timeouts_remaining', 'home_team', 'result']

nfl_df = pbp[cols].dropna()

# Create target variable — did possession team win?
nfl_df = nfl_df.copy()
nfl_df['win'] = (nfl_df['result'] > 0).astype(int)

print(f"Shape: {nfl_df.shape}")
print(f"Win rate: {nfl_df['win'].mean():.1%}")
print(nfl_df.head())

Shape: (165790, 12)
Win rate: 55.0%
           game_id posteam defteam  game_seconds_remaining  \
2  2022_01_BAL_NYJ     NYJ     BAL                  3596.0   
3  2022_01_BAL_NYJ     NYJ     BAL                  3569.0   
4  2022_01_BAL_NYJ     NYJ     BAL                  3565.0   
5  2022_01_BAL_NYJ     NYJ     BAL                  3541.0   
6  2022_01_BAL_NYJ     NYJ     BAL                  3533.0   

   score_differential  down  ydstogo  yardline_100  \
2                 0.0   1.0     10.0          78.0   
3                 0.0   1.0     10.0          59.0   
4                 0.0   2.0     10.0          59.0   
5                 0.0   3.0      5.0          54.0   
6                 0.0   4.0     15.0          64.0   

   posteam_timeouts_remaining home_team  result  win  
2                         3.0       NYJ     -15    0  
3                         3.0       NYJ     -15    0  
4                         3.0       NYJ     -15    0  
5                         3.0       NYJ     -1

In [4]:
import plotly.express as px

# Win rate by score differential
nfl_df['score_diff_bucket'] = nfl_df['score_differential'].clip(-35, 35)
win_by_score = nfl_df.groupby('score_diff_bucket')['win'].mean().reset_index()

fig = px.line(win_by_score, 
              x='score_diff_bucket', 
              y='win',
              title='NFL win rate by score differential',
              labels={'score_diff_bucket': 'Score differential', 'win': 'Win rate'})
fig.show()

# Win rate by time remaining
nfl_df['time_bucket'] = (nfl_df['game_seconds_remaining'] // 60).astype(int)
win_by_time = nfl_df.groupby('time_bucket')['win'].mean().reset_index()

fig2 = px.line(win_by_time,
               x='time_bucket',
               y='win',
               title='NFL win rate by minutes remaining',
               labels={'time_bucket': 'Minutes remaining', 'win': 'Win rate'})
fig2.show()

In [5]:
# Feature engineering — create the inputs for the model
nfl_df = nfl_df.copy()

# The key interaction — score diff means different things at different times
nfl_df['score_x_time'] = nfl_df['score_differential'] * nfl_df['game_seconds_remaining']

# Is the possession team the home team?
nfl_df['is_home'] = (nfl_df['posteam'] == nfl_df['home_team']).astype(int)

# Final feature list
features = [
    'score_differential',
    'game_seconds_remaining',
    'score_x_time',
    'down',
    'ydstogo',
    'yardline_100',
    'posteam_timeouts_remaining',
    'is_home'
]

X = nfl_df[features]
y = nfl_df['win']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature summary:")
print(X.describe().round(2))

Features shape: (165790, 8)
Target shape: (165790,)

Feature summary:
       score_differential  game_seconds_remaining  score_x_time       down  \
count           165790.00               165790.00     165790.00  165790.00   
mean                -1.36                 1719.01      -1964.80       2.00   
std                 10.13                 1048.11      14035.01       1.01   
min                -56.00                    0.00     -80955.00       1.00   
25%                 -7.00                  804.00      -8841.00       1.00   
50%                  0.00                 1800.00          0.00       2.00   
75%                  4.00                 2606.00       4172.00       3.00   
max                 50.00                 3600.00      75600.00       4.00   

         ydstogo  yardline_100  posteam_timeouts_remaining   is_home  
count  165790.00     165790.00                   165790.00  165790.0  
mean        8.47         50.15                        2.61       0.5  
std         4.

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

Training rows: 132632
Testing rows: 33158
